# Original YOLO12-Small - CH-RDD2022 (Kaggle)

Baseline ini memakai YOLO12-Small resmi tanpa YAML custom, attention, GhostConv, ECA, EMA, SPD-Conv, atau perubahan source code. Bobot awal adalah yolo12s.pt resmi.

Gunakan notebook ini dengan split, seed, augmentasi, dan hyperparameter yang sama dengan setiap varian modifikasi.

In [ ]:
# 1. Install Ultralytics resmi. Aktifkan GPU dan Internet pada Kaggle.
import json
import platform
import subprocess
import sys
import zipfile
from pathlib import Path

WORKDIR = Path('/kaggle/working')
ULTRALYTICS_VERSION = '8.4.0'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'ultralytics=={ULTRALYTICS_VERSION}'], check=True)

import torch
import ultralytics
from ultralytics import YOLO

def log_section(title: str) -> None:
    print(f'\n{"=" * 88}\n{title}\n{"=" * 88}')

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
log_section('OFFICIAL YOLO12S ENVIRONMENT')
print(f'Python      : {platform.python_version()}')
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'CUDA ready  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Dataset, hyperparameter, dan model official tanpa modifikasi.
DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ch-rdd2022/datasets-china-split')
DATA_YAML = WORKDIR / 'ch_rdd2022.yaml'
MODEL_WEIGHTS = 'yolo12s.pt'
EXPERIMENT_NAME = 'yolo12s_original_ch_rdd2022_pretrained'
RUNS_DIR = WORKDIR / 'runs'

# Samakan dengan varian GhostConv+EMA32: batch fisik 16, nominal batch 64.
EPOCHS, IMGSZ, BATCH, NBS = 160, 640, 16, 64
OPTIMIZER, LR0, MOMENTUM, WEIGHT_DECAY = 'SGD', 0.01, 0.937, 0.0005
PATIENCE, WORKERS, SEED = 0, 2, 42

DATA_YAML.write_text(f'''path: {DATA_ROOT}
train: train/images
val: val/images
test: test/images

nc: 5
names:
  0: D00
  1: D10
  2: D20
  3: D40
  4: Repair
''', encoding='utf-8')
assert DATA_ROOT.exists(), f'Dataset path tidak ditemukan: {DATA_ROOT}'

log_section('ORIGINAL YOLO12S MODEL INFO')
model = YOLO(MODEL_WEIGHTS)
print('Architecture : official YOLO12s; no custom YAML or module')
print('Pretrained   :', MODEL_WEIGHTS)
print('Parameters   :', f'{sum(p.numel() for p in model.model.parameters()):,}')
print(model.model)


In [ ]:
# 3. Fine-tune pretrained original YOLO12s. Head akan otomatis disesuaikan menjadi 5 kelas.
log_section('TRAINING STARTED - ORIGINAL YOLO12S')
print(f'epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, nbs={NBS}, optimizer={OPTIMIZER}, lr0={LR0}, seed={SEED}')
model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, nbs=NBS, device=DEVICE, workers=WORKERS,
    project=str(RUNS_DIR), name=EXPERIMENT_NAME, exist_ok=True, pretrained=True, optimizer=OPTIMIZER,
    lr0=LR0, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, cos_lr=False, patience=PATIENCE,
    seed=SEED, plots=True, verbose=True,
)
RUN_DIR, BEST_PT, LAST_PT = Path(model.trainer.save_dir), Path(model.trainer.best), Path(model.trainer.last)
print(f'Run directory: {RUN_DIR}')
print(f'Best weights : {BEST_PT}')


In [ ]:
# 4. Evaluasi best checkpoint dan buat ZIP hasil.
log_section('BEST CHECKPOINT EVALUATION')
best_model = YOLO(str(BEST_PT))

def metric_summary(metrics) -> dict:
    return {'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
            'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map),
            'save_dir': str(metrics.save_dir)}

val_metrics = best_model.val(data=str(DATA_YAML), split='val', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                             project=str(RUNS_DIR), name=f'{EXPERIMENT_NAME}_val', exist_ok=True, plots=True)
EVALUATION_REPORT = {'validation': metric_summary(val_metrics)}
test_labels = DATA_ROOT / 'test' / 'labels'
if test_labels.exists() and any(test_labels.glob('*.txt')):
    test_metrics = best_model.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                                  project=str(RUNS_DIR), name=f'{EXPERIMENT_NAME}_test', exist_ok=True, plots=True)
    EVALUATION_REPORT['test'] = metric_summary(test_metrics)
    TEST_OUTPUT_DIR = Path(test_metrics.save_dir)
else:
    predictions = best_model.predict(source=str(DATA_ROOT / 'test' / 'images'), imgsz=IMGSZ, device=DEVICE,
                                    conf=0.25, save=True, save_txt=True, project=str(RUNS_DIR),
                                    name=f'{EXPERIMENT_NAME}_test_predictions', exist_ok=True)
    TEST_OUTPUT_DIR = Path(predictions[0].save_dir) if predictions else RUNS_DIR
    EVALUATION_REPORT['test'] = {'status': 'labels unavailable; prediction only', 'save_dir': str(TEST_OUTPUT_DIR)}

METRICS_JSON = WORKDIR / f'{EXPERIMENT_NAME}_evaluation_metrics.json'
METRICS_JSON.write_text(json.dumps(EVALUATION_REPORT, indent=2), encoding='utf-8')
RUN_CONFIG = WORKDIR / f'{EXPERIMENT_NAME}_config.json'
RUN_CONFIG.write_text(json.dumps({
    'architecture': 'official YOLO12s without modification', 'weights': MODEL_WEIGHTS,
    'dataset_root': str(DATA_ROOT), 'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH, 'nbs': NBS,
    'optimizer': OPTIMIZER, 'lr0': LR0, 'momentum': MOMENTUM, 'weight_decay': WEIGHT_DECAY, 'seed': SEED,
    'best_checkpoint': str(BEST_PT), 'last_checkpoint': str(LAST_PT),
}, indent=2), encoding='utf-8')

ZIP_PATH = WORKDIR / f'{EXPERIMENT_NAME}_results.zip'
def add_to_zip(archive: zipfile.ZipFile, path: Path) -> int:
    if not path.exists():
        return 0
    files = [path] if path.is_file() else [item for item in path.rglob('*') if item.is_file()]
    for file_path in files:
        archive.write(file_path, file_path.relative_to(WORKDIR))
    return len(files)

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    count = sum(add_to_zip(archive, Path(item)) for item in (
        RUN_DIR, Path(val_metrics.save_dir), TEST_OUTPUT_DIR, DATA_YAML, RUN_CONFIG, METRICS_JSON,
    ))
print(json.dumps(EVALUATION_REPORT, indent=2))
print(f'ZIP created : {ZIP_PATH} ({count} files)')
from IPython.display import FileLink, display
display(FileLink(ZIP_PATH))
